In [37]:
import pdfplumber
import json
import re
from datetime import datetime
from typing import Dict, List, Optional
import pandas as pd
from pathlib import Path

## Импорт отчетностей

In [38]:
gazp = "C:/Users/Сергей Кравцов/Desktop/Morgan Co/Imperial Hedge Fund/data/processed/СБЕР/reporting_q4_aolql6na_2025-5-10.pdf"
ticker = "SBER"

## Чтение отчетностей

In [39]:
def extract_all_text(pdf_path):
    """Извлекает текст со всех страниц PDF."""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_text = page.extract_text() or ''
            if page_text.strip():  # пропускаем пустые страницы
                text += f"\n{'='*60}\nСТРАНИЦА {i+1}\n{'='*60}\n"
                text += page_text
    return text

# Извлекаем текст
full_text = extract_all_text(gazp)
print(f"✅ Извлечено {len(full_text)} символов")
print(f"📄 Количество страниц: {full_text.count('СТРАНИЦА')}")

✅ Извлечено 14505 символов
📄 Количество страниц: 6


## Сохранение промпта для отправки ИИ

In [40]:
prompt = f"""
Ты — профессиональный финансовый аналитик. Извлеки из отчётности компании {ticker} все финансовые показатели.

Вот текст отчётности (только нужные страницы):

{full_text[:15000]}

Извлеки следующие показатели и верни их в формате JSON:

1. Отчёт о прибылях и убытках:
   - revenue (выручка)
   - cost_of_revenue (себестоимость)
   - gross_profit (валовая прибыль)
   - operating_income (прибыль от продаж)
   - ebitda
   - net_income (чистая прибыль)
   - interest_expense (проценты к уплате)
   - taxes (налоги)
   - depreciation (амортизация)

2. Баланс:
   - total_assets (всего активов)
   - current_assets (оборотные активы)
   - cash (денежные средства)
   - inventory (запасы)
   - receivables (дебиторская задолженность)
   - total_equity (собственный капитал)
   - total_liabilities (обязательства всего)
   - short_term_debt (краткосрочные займы)
   - long_term_debt (долгосрочные займы)
   - total_debt (всего долг)

3. Отчёт о движении денежных средств:
   - operating_cash_flow (операционный денежный поток)
   - free_cash_flow (свободный денежный поток)
   - capex (капитальные вложения)
   - dividends_paid (выплаченные дивиденды)

Если показатель не найден, поставь null.

Ответь ТОЛЬКО JSON, без пояснений. Используй российскую валюту (рубли).
"""

In [41]:
Path("prompts").mkdir(exist_ok=True)
with open(f"prompts/{ticker}_prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt)
print(f"Промпт сохранён в prompts/{ticker}_prompt.txt")

Промпт сохранён в prompts/SBER_prompt.txt
